In [3]:
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import pandas as pd
import numpy as np
import pytz
import re
import matplotlib.dates as mdates
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas


In [4]:
# # populate total station count for all SSIDs per radio

def calculate_total_scount(df):

    # Get unique wlan.ta and radio_number pairs
    radio_1_transmitters = df[df["radio_number"] == 1]["wlan.ta"].drop_duplicates().tolist()
    radio_2_transmitters = df[df["radio_number"] == 2]["wlan.ta"].drop_duplicates().tolist()

    # Initialize the total_scount column
    if "total_scount" not in df.columns:
        df["total_scount"] = 0


    # Iterate over the rows to calculate total_scount
    for i in range(len(df)):
        current_ta = df.at[i, "wlan.ta"]
        current_scount = df.at[i, "wlan.qbss.scount"]
        current_radio = df.at[i, "radio_number"]

        # set transmitter list based on radio
        if current_radio == 1:
             radio_transmitters = radio_1_transmitters.copy()
        elif current_radio == 2:
             radio_transmitters = radio_2_transmitters.copy()
        else:
            print("error - no radio")

        # remove the current line radio from the list
        radio_transmitters.remove(current_ta)
        
        #create loop for every entry in radio_x_transmitters
        for transmitter in radio_transmitters:
             #find the next value for this transmitter and add it to current_scount

            # Look for a next record where wlan.ta matches the one in the list
            j = i + 1  # Start with the next row
            while j < (len(df)):
                next_ta = df.at[j, "wlan.ta"]
                if next_ta == transmitter:
                    # Found a record matching the ta
                    next_scount = df.at[j, "wlan.qbss.scount"]
                    current_scount += next_scount
                    break
                j += 1  # Go to the next row

            # If no valid next record was found, look backwards
            if j >= (len(df)):
                k = i - 1  # Start with the previous row
                while k > 0:
                    prev_ta = df.at[k, "wlan.ta"]
                    if prev_ta == transmitter:
                        # Found the next record with the same wlan.ta_int
                        prev_scount = df.at[k, "wlan.qbss.scount"]
                        current_scount += prev_scount
                        break
                    k -= 1
            
            # If no next record exists, keep total_scount as current_scount
        #update the value after all is added
        df.at[i, "total_scount"] = current_scount
    
    # Ensure total_scount is of type integer
    df["total_scount"] = df["total_scount"].astype(int)

    return df

In [5]:
#  Define the function to generate the plot and save it as a file
#inputs are dataframes filtered to the radio, and further filtered for each SSID
def plot_channel_stations(radio_df, eduroam_df, byu_wifi_df, AP_name, start_time, end_time):
    #print("plot_channel_stations")
    plt.figure(figsize=(14, 7))  # Example size: 12 inches wide, 6 inches tall
    # Add a horizontal line at value 70 with 20% opacity
    plt.axhline(y=70, color='red', linestyle='-', alpha=0.2, zorder=-1)

    # Access the first record and get the value of 'radio_number'
    plot_radio_number = radio_df.iloc[0]["radio_number"]
    
    # Plot wlan.qbss.cu normalized as percentages
    plt.plot(radio_df["aruba_erm.time"], (radio_df["wlan.qbss.cu"] / 255) * 100, label="Channel Utilization (%)")

    # Fill the area below the line
    plt.fill_between(radio_df["aruba_erm.time"], (radio_df["wlan.qbss.cu"] / 255) * 100)


    # Plot total_scount
    plt.plot(radio_df["aruba_erm.time"], radio_df["total_scount"], label="Total Station Count")

    # Plot total_scount for "eduroam"
    #plt.plot(eduroam_df["aruba_erm.time"], eduroam_df["wlan.qbss.scount"], label="eduroam Station Count")

    # Plot total_scount for "BYU-WiFi"
    #plt.plot(byu_wifi_df["aruba_erm.time"], byu_wifi_df["wlan.qbss.scount"], label="BYU-WiFi Station Count")

    
    
    # Plot total_adc
    #plt.plot(radio_df["aruba_erm.time"], radio_df["wlan.qbss.adc"] / 32767 * 100, label="ADC (%)")

    #plot adc% + cu
    #adc updates less frequently than cu, but seems to be well correlated
    #plt.plot(radio_df["aruba_erm.time"], 100 - ((radio_df["wlan.qbss.adc"] / 32767 * 100) + (radio_df["wlan.qbss.cu"] / 255 * 100)), label="ADC + CU")



    # Add labels and title
    plt.xlabel(f"{start_time}   to   {end_time}", fontsize=14)
    plt.ylabel("Values", fontsize=14)
    plt.title(f"{AP_name} Radio {plot_radio_number}", fontsize=16)

    # Set the y-axis limits from 0 to 100
    plt.ylim(0, 100)

    # Format x-axis to show only time (hour:minute)
    #showing GMT, this is broken
    # time_formatter = mdates.DateFormatter("%H:%M")  # Format: Hour:Minute
    # plt.gca().xaxis.set_major_formatter(time_formatter)

    # Rotate x-axis labels for better visibility
    #plt.xticks(rotation=45)

    # Add a legend
    plt.legend()



    # Save the plot as a PNG file (you could also save as PDF)
    plot_file = "Output/" + AP_name + "_" + start_time.strftime("%Y-%m-%d_%H-%M") + "_R" + str(plot_radio_number) + ".png"
    #print(plot_file)
    ###temp### plt.savefig(plot_file)
    #print(plot_file)
    # Display the plot
    #plt.show()
    return plot_file

In [6]:
# Function to perform the analysis for a given radio DataFrame
def analyze_radio_group(radio_df, radio_number):
    #print("analyze_radio_group")
    # Ensure the DataFrame is sorted by timestamp
    radio_df = radio_df.sort_values(by="aruba_erm.time").reset_index(drop=True)

    # Count the total number of data points
    total_data_points = len(radio_df)

    # Calculate the greatest value of wlan.qbss.cu
    greatest_cutil = ((radio_df["wlan.qbss.cu"].max() / 255) * 100)

    # Calculate the least value of wlan.qbss.cu
    least_cutil = ((radio_df["wlan.qbss.cu"].min() / 255) * 100)

    # Count the number of data points where wlan.qbss.cu is greater than 178
    greater_than_178_count = (radio_df["wlan.qbss.cu"] > 178).sum()

    # Calculate the percentage of data points where wlan.qbss.cu is greater than 178
    percentage_greater_than_178 = (greater_than_178_count / total_data_points) * 100

    # Median wlan.qbss.cu
    median_cutil = ((radio_df["wlan.qbss.cu"].median() / 255) * 100)

    # Calculate the greatest value of wlan.qbss.scount
    greatest_scount = radio_df["total_scount"].max()

    # Calculate the median value of wlan.qbss.scount
    median_scount = radio_df["total_scount"].median()

    # Calculate the least value of wlan.qbss.scount
    least_scount = radio_df["total_scount"].min()

    # Print the results
    print()
    print(f"--- Radio Number {radio_number} ---")
    print(f"Total beacons: {total_data_points}")
    print(f"Number of beacons with high channel utilization: {greater_than_178_count}")
    print(f"Percentage of beacons with high channel utilization: {percentage_greater_than_178:.2f}%")
    print(f"Highest channel utilization: {greatest_cutil:.2f}%")
    print(f"Median channel utilization: {median_cutil:.2f}%")
    print(f"Lowest channel utilization: {least_cutil:.2f}%")
    print(f"Highest station count: {greatest_scount:.0f}")
    print(f"Median station count: {median_scount:.0f}")
    print(f"Lowest station count: {least_scount:.0f}")

    # Create a boolean mask for wlan.qbss.cu > 178
    condition = radio_df["wlan.qbss.cu"] > 178

    # Assign group numbers for consecutive periods where condition is True
    radio_df["group"] = (condition != condition.shift()).cumsum() * condition

    # Filter groups where wlan.qbss.cu > 178
    valid_groups = radio_df[radio_df["group"] > 0]

    # Check if there are valid groups (i.e., groups where wlan.qbss.cu > 178)
    if valid_groups.empty:
        print(f"No records found where wlan.qbss.cu > 178 for radio_number {radio_number}.")
        longest_duration = 0
    else:
       # Find the start and end of each group
        group_details = valid_groups.groupby("group").agg(
            first_index=("aruba_erm.time", "idxmin"),
            last_index=("aruba_erm.time", "idxmax")
        )

        # Extract the first and last timestamps directly
        group_details["first_in_sequence_time"] = valid_groups.groupby("group")["aruba_erm.time"].min()
        group_details["end_time"] = valid_groups.groupby("group")["aruba_erm.time"].max()

        # Calculate the duration between the start and end times, and add 1 second 
        # Aruba updates beacons once per 10, which is also the rate that we have captured. 1.024 is the time the first beacon represents
        group_details["duration"] = (group_details["end_time"] - group_details["first_in_sequence_time"]).dt.total_seconds() + 1.024

        # Find the group with the longest duration
        longest_group = group_details["duration"].idxmax()
        longest_row = group_details.loc[longest_group]

        # Extract details of the longest duration
        longest_duration = longest_row["duration"]
        longest_start_time = longest_row["first_in_sequence_time"]
        longest_end_time = longest_row["end_time"]

        # Print results for the specific radio_number
        print(f"Longest consecutive duration of high channel use: {longest_duration:.2f} seconds")
        print(f"First beacon of sequence: {longest_start_time}")
        print(f"Last beacon of sequence: {longest_end_time}")

    return total_data_points, greater_than_178_count, percentage_greater_than_178, greatest_cutil, median_cutil, least_cutil, greatest_scount, median_scount, least_scount, longest_duration

In [8]:
# Receives the information for each AP and class time information
# Reads the file and processes the information for that time/AP
def graph(AP_name, start_time, end_time, location, course, enrolled, capacity, ap_count, ap_df, class_summary_df):

    print(f"AP name: {AP_name}")

    # this is improved by loading the file once per AP instead of once per class
    # Define the base directory for the files
    #base_dir = "Data"
    #file_name = AP_name + ".parquet"

    # Load the data for the AP file into a dataframe
    #df = pd.read_parquet(
    #    f"{base_dir}/{file_name}",
    #    engine="pyarrow",
    #)

    # copy the AP df for manipulation
    filtered_df = ap_df.copy()

    # time column
    filtered_df.reset_index(inplace=True) #reset index so that the timestamp is in a column instead

    # Fix timestamp consistency
    # Localize only if the timestamps are naive
    if filtered_df["aruba_erm.time"].dt.tz is None:
        filtered_df["aruba_erm.time"] = filtered_df["aruba_erm.time"].dt.tz_localize("UTC")

    # Convert to UTC-7 (this can always be done after localization)
    filtered_df["aruba_erm.time"] = filtered_df["aruba_erm.time"].dt.tz_convert("Etc/GMT+7")

    # Filter the DataFrame by the date/time range
    filtered_df = filtered_df[(filtered_df['aruba_erm.time'] >= start_time) & (filtered_df['aruba_erm.time'] <= end_time)]

    # return if there's no data
    if filtered_df.empty:
        print(f"No data available for {AP_name} from {start_time} to {end_time}.")
        return

    # Ensure the DataFrame is sorted by timestamp
    filtered_df = filtered_df.sort_values(by="aruba_erm.time").reset_index(drop=True)


    # Convert wlan.qbss.scount to integer, handling NaN values
    filtered_df["wlan.qbss.scount"] = filtered_df["wlan.qbss.scount"].fillna(0).astype(int)
    filtered_df["wlan.qbss.cu"] = filtered_df["wlan.qbss.cu"].fillna(0).astype(int)
    filtered_df["wlan.qbss.adc"] = filtered_df["wlan.qbss.adc"].fillna(0).astype(int)

    # create wlan.ta_int column
    # Function to convert hex MAC address string to integer
    def mac_to_int(mac):
        return int(mac.replace(":", ""), 16)

    # Apply the conversion to the wlan.ta column
    filtered_df["wlan.ta_int"] = filtered_df["wlan.ta"].apply(mac_to_int)
    
    

    # Check Radio and SSID count
    # Verify all values in wlan.vs.aruba.ap_name are the same
    if filtered_df["wlan.vs.aruba.ap_name"].nunique() != 1:
        raise ValueError("Values in wlan.vs.aruba.ap_name are not all the same.")
    #else:
        #wlan_ap_name_value = filtered_df["wlan.vs.aruba.ap_name"].iloc[0]
        #print(f"Data from {wlan_ap_name_value}")
        


    # Count unique transmit addresses and corresponding SSIDs
    # ensures that data is clean and shows multi-radio APs
    # Group by wlan.ta and collect the corresponding wlan.ssid values
    bssid_info = (
        filtered_df.groupby("wlan.ta")["wlan.ssid"]
        .unique()  # Get unique SSIDs for each wlan.ta
        .reset_index()
        .rename(columns={"wlan.ta": "transmit_address", "wlan.ssid": "SSIDs"})
    )

    # Count the number of unique BSSIDs
    unique_bssid_count = bssid_info["transmit_address"].nunique()


    # add a column for unique radio number
    # Create the 'radio_number' column
    filtered_df = filtered_df.sort_values("wlan.ta_int").reset_index(drop=True)

    # Initialize the 'radio_number' column
    radio_number = 1
    radio_numbers = [radio_number]

    # Assign radio numbers based on 'wlan.ta_int'
    # would probably be better to use channel number, but this works
    for i in range(1, len(filtered_df)):
        # Check if the difference between consecutive wlan.ta_int values is greater than 1
        if abs(filtered_df.loc[i, "wlan.ta_int"] - filtered_df.loc[i - 1, "wlan.ta_int"]) > 1:
            radio_number += 1  # Increment radio number if the condition is met
        radio_numbers.append(radio_number)

    # Add the radio_number column to the DataFrame
    filtered_df["radio_number"] = radio_numbers

    # Ensure the DataFrame is sorted by timestamp
    filtered_df = filtered_df.sort_values(by="aruba_erm.time").reset_index(drop=True)


    # Apply the function to calculate total_scount
    filtered_df = calculate_total_scount(filtered_df)

    # Verify the result
    #print(filtered_df[["wlan.ta_int", "wlan.qbss.scount", "total_scount"]].head())
    #print(filtered_df.dtypes)


    # Display the updated DataFrame
    #print(filtered_df.head())

    #print number of radios
    radio_count = filtered_df['radio_number'].max()
    #print(f"Number or 5GHz radios: {filtered_df['radio_number'].max()}")
    print(f"Number or 5GHz radios: {radio_count}")


    # Output the BSSID results
    print(f"Number of BSSIDs: {unique_bssid_count}")
    print("List of addresses and their corresponding SSIDs:")
    print(bssid_info)


    # Filter the DataFrame for records where radio_number is 1
    filtered_radio_1_df = filtered_df[filtered_df["radio_number"] == 1]
    # Filter the data for "eduroam" and "BYU-WiFi" SSIDs
    eduroam_radio_1_df = filtered_df[(filtered_df["wlan.ssid"] == "eduroam") & (filtered_df["radio_number"] == 1)]
    byu_wifi_radio_1_df = filtered_df[(filtered_df["wlan.ssid"] == "BYU-WiFi") & (filtered_df["radio_number"] == 1)]

    #  Filter the DataFrame for records where radio_number is 2
    filtered_radio_2_df = filtered_df[filtered_df["radio_number"] == 2]

    # Run the analysis for radio_number 1 (using filtered_radio_1_df)
    if not filtered_radio_1_df.empty:
        total_beacons, high_cu_beacons, percent_high_cu, highest_cu, median_cu, lowest_cu, high_scount, median_scount, low_scount, longest_duration_high_cu = analyze_radio_group(filtered_radio_1_df, 1)
        # Plot for the first radio
        plot_file = plot_channel_stations(filtered_radio_1_df, eduroam_radio_1_df, byu_wifi_radio_1_df, AP_name, start_time, end_time)
        #print(plot_file)
        #update totals summary
        class_summary_df.loc[len(class_summary_df)] = [
        course, location, AP_name, 1, start_time, end_time,
        capacity, enrolled, ap_count, radio_count, total_beacons,
        high_cu_beacons, percent_high_cu, highest_cu, median_cu, lowest_cu,
        high_scount, median_scount, low_scount, longest_duration_high_cu, plot_file
        ]

    # Run the analysis for radio_number 2 (using filtered_radio_2_df) only if it's not empty
    if not filtered_radio_2_df.empty:
        total_beacons, high_cu_beacons, percent_high_cu, highest_cu, median_cu, lowest_cu, high_scount, median_scount, low_scount, longest_duration_high_cu = analyze_radio_group(filtered_radio_2_df, 2)


        # Filter the data for "eduroam" and "BYU-WiFi" SSIDs
        eduroam_radio_2_df = filtered_df[(filtered_df["wlan.ssid"] == "eduroam") & (filtered_df["radio_number"] == 2)]
        byu_wifi_radio_2_df = filtered_df[(filtered_df["wlan.ssid"] == "BYU-WiFi") & (filtered_df["radio_number"] == 2)]
        # Plot for the second radio
        plot_file = plot_channel_stations(filtered_radio_2_df, eduroam_radio_2_df, byu_wifi_radio_2_df, AP_name, start_time, end_time)
        #print(plot_file)
        #update totals summary
        class_summary_df.loc[len(class_summary_df)] = [
        course, location, AP_name, 2, start_time, end_time,
        capacity, enrolled, ap_count, radio_count, total_beacons,
        high_cu_beacons, percent_high_cu, highest_cu, median_cu, lowest_cu,
        high_scount, median_scount, low_scount, longest_duration_high_cu, plot_file
        ]
    else:
        print(f"{AP_name} has only one 5GHz radio")

    #print()
    
    # return totals
    return class_summary_df


In [9]:
#read in sample files
# this can be done after pickle files have been created from samples.py
sample_classes_df = pd.read_pickle('sample_classes.pkl')
sample_aps_df = pd.read_pickle('sample_aps.pkl')
sample_classrooms_df = pd.read_pickle('sample_classrooms.pkl')

In [ ]:
print(sample_classes_df.head())


In [ ]:

print(sample_aps_df.head())


In [ ]:
print(sample_classrooms_df)

In [10]:

#create a separate dataframe for aggregate information
# Define column names and data types
columns = {
    "subject": str,
    "location": str,
    "AP_hostname": str,
    "radio_number": int,
    "start_time": "datetime64[ns]",
    "end_time": "datetime64[ns]",
    "capacity": int,
    "enrolled": int,
    "AP_count": int,
    "radio_count": int,
    "total_beacons": int,
    "high_cu_beacons": int,
    "percent_high_cu": float,
    "highest_cu": int,
    "median_cu": int,
    "lowest_cu": int,
    "high_scount": int,
    "median_scount": int,
    "low_scount": int,
    "longest_duration_high_cu": float,
    "graph_filename": str
}

# Create an empty DataFrame with specified columns and data types
class_summary_df = pd.DataFrame({col: pd.Series(dtype=dtype) for col, dtype in columns.items()})

# Display the empty DataFrame
print(class_summary_df)


Empty DataFrame
Columns: [subject, location, AP_hostname, radio_number, start_time, end_time, capacity, enrolled, AP_count, radio_count, total_beacons, high_cu_beacons, percent_high_cu, highest_cu, median_cu, lowest_cu, high_scount, median_scount, low_scount, longest_duration_high_cu, graph_filename]
Index: []

[0 rows x 21 columns]


In [ ]:
# run analysis for only one location
# Specify the target location
target_location = "TMCB-155A"
# Define the base directory for the files
base_dir = "Data"

# Initialize variables to ensure it's defined
ap_df = pd.DataFrame()
AP_name = ""

# Loop through each row in sample_classes_df
for index, class_row in sample_classes_df.iterrows():
    location = class_row['location']
    
    # Skip rows that don't match the target location
    if location != target_location:
        continue
    
    start_time = class_row['start_time']
    end_time = class_row['end_time']
    course = class_row['course']
    enrolled = class_row['enrolled']
    capacity = class_row['capacity']
    ap_count = sample_classrooms_df.loc[sample_classrooms_df['Location'] == location, 'ap_count'].iloc[0]


    print("\n")
    print(f"Subject: {course}")
    print(f"Location: {location}")
    print(f"Start Time: {start_time}")
    print(f"End Time: {end_time}")
    print(f"Capacity: {capacity}")
    print(f"Enrolled: {enrolled}")
    print(f"Number of APs: {ap_count}")
    print()
    
    # Sub loop for each matching AP in sample_aps_df
    for ap_index, ap_row in sample_aps_df.iterrows():
        building_room = ap_row['Building-Room']
        
        # Check if the building-room matches the location
        if building_room == location:
            
            if AP_name != ap_row['Hostname']:
                AP_name = ap_row['Hostname']
                file_name = AP_name + ".parquet"
                # Load the data for the AP file into a dataframe
                # this could be improved by loading the file once per AP instead of once per class
                ap_df = pd.read_parquet(
                    f"{base_dir}/{file_name}",
                    engine="pyarrow",
                )

            
            # Call the graph function with the variables
            class_summary_df = graph(AP_name, start_time, end_time, location, course, enrolled, capacity, ap_count, ap_df, class_summary_df)

            


In [ ]:
# loop for each class row, iterate through each AP at that location
# this will go through ALL data and take a while
# need to remove the non-5xx APs from the sample

# Define the base directory for the files
base_dir = "Data"

# Initialize variables to ensure it's defined
ap_df = pd.DataFrame()
AP_name = ""

# Loop through each row in sample_classes_df
for index, class_row in sample_classes_df.iterrows():
    location = class_row['location']
    start_time = class_row['start_time']
    end_time = class_row['end_time']
    course = class_row['course']
    enrolled = class_row['enrolled']
    capacity = class_row['capacity']
    ap_count = sample_classrooms_df.loc[sample_classrooms_df['Location'] == location, 'ap_count'].iloc[0]


    print("\n")
    print(f"Subject: {course}")
    print(f"Location: {location}")
    print(f"Start Time: {start_time}")
    print(f"End Time: {end_time}")
    print(f"Capacity: {capacity}")
    print(f"Enrolled: {enrolled}")
    print(f"AP count {ap_count}")
    print()
    
    # Sub loop for each matching AP in sample_aps_df
    for ap_index, ap_row in sample_aps_df.iterrows():
        building_room = ap_row['Building-Room']
        
        # Check if the building-room matches the location
        if building_room == location:
                
            if AP_name != ap_row['Hostname']:
                AP_name = ap_row['Hostname']
                file_name = AP_name + ".parquet"
                # Load the data for the AP file into a dataframe
                # this could be improved by loading the file once per AP instead of once per class
                ap_df = pd.read_parquet(
                    f"{base_dir}/{file_name}",
                    engine="pyarrow",
                )
            
            # Call the graph function with the variables
            graph(AP_name, start_time, end_time, location, course, enrolled, capacity, ap_count, ap_df, class_summary_df)

# write summary dataframe to a file so it can be analyzed further later
class_summary_df.to_pickle('class_summary.pkl')


            


In [4]:
class_summary_df = pd.read_pickle('class_summary.pkl')
class_summary_df.head()

,subject,location,AP_hostname,radio_number,start_time,end_time,capacity,enrolled,AP_count,radio_count,...,high_cu_beacons,percent_high_cu,highest_cu,median_cu,lowest_cu,high_scount,median_scount,low_scount,longest_duration_high_cu,graph_filename
0,NURS 296,BNSN-W003,P-BNSN-W003-AP-01-W131,1,2024-11-04 08:30:00-07:00,2024-11-04 09:50:00-07:00,25,8,1,1,...,4,0.042571,94.509804,2.745098,0.392157,14,7.0,0,2.033519,Output/P-BNSN-W003-AP-01-W131_2024-11-04_08-30...
1,CHEM 105,BNSN-W003,P-BNSN-W003-AP-01-W131,1,2024-11-05 08:00:00-07:00,2024-11-05 08:50:00-07:00,25,21,1,1,...,6,0.101215,82.745098,5.098039,1.176471,30,28.0,9,3.033024,Output/P-BNSN-W003-AP-01-W131_2024-11-05_08-00...
2,CHEM 105,BNSN-W003,P-BNSN-W003-AP-01-W131,1,2024-11-05 09:00:00-07:00,2024-11-05 09:50:00-07:00,25,22,1,1,...,46,0.857569,90.980392,8.235294,1.176471,33,29.0,17,4.039982,Output/P-BNSN-W003-AP-01-W131_2024-11-05_09-00...
3,CHEM 105,BNSN-W003,P-BNSN-W003-AP-01-W131,1,2024-11-05 10:00:00-07:00,2024-11-05 10:50:00-07:00,25,21,1,1,...,2,0.033784,70.588235,6.666667,1.176471,28,26.0,13,1.024129,Output/P-BNSN-W003-AP-01-W131_2024-11-05_10-00...
4,CHEM 106,BNSN-W003,P-BNSN-W003-AP-01-W131,1,2024-11-05 12:00:00-07:00,2024-11-05 12:50:00-07:00,25,14,1,1,...,4,0.067659,90.588235,3.921569,0.784314,18,14.0,11,2.029625,Output/P-BNSN-W003-AP-01-W131_2024-11-05_12-00...


In [13]:
class_summary_df.dtypes

subject                                        object
location                                       object
AP_hostname                                    object
radio_number                                    int64
start_time                  datetime64[ns, Etc/GMT+7]
end_time                    datetime64[ns, Etc/GMT+7]
capacity                                        int64
enrolled                                        int64
AP_count                                        int64
radio_count                                     int64
total_beacons                                   int64
high_cu_beacons                                 int64
percent_high_cu                               float64
highest_cu                                    float64
median_cu                                     float64
lowest_cu                                     float64
high_scount                                     int64
median_scount                                 float64
low_scount                  

In [ ]:
#class_summary_df = class_summary_df.sort_values(by="percent_high_cu", ascending=False)
#radio count seems to be broken
class_summary_df = class_summary_df.sort_values(by="total_beacons")
class_summary_df.head(20)

,subject,location,AP_hostname,radio_number,start_time,end_time,capacity,enrolled,AP_count,radio_count,...,high_cu_beacons,percent_high_cu,highest_cu,median_cu,lowest_cu,high_scount,median_scount,low_scount,longest_duration_high_cu,graph_filename
5192,MSB 380,TNRB-151,P-TNRB-151-AP-02-140B,1,2024-11-06 11:00:00-07:00,2024-11-06 12:15:00-07:00,344,334,4,1,...,0,0.0,22.352941,22.352941,22.352941,12,12.0,12,0.0,Output/P-TNRB-151-AP-02-140B_2024-11-06_11-00_...
4089,ECON 440,MARB-B124,P-MARB-B124-AP-01-B111,1,2024-11-04 09:30:00-07:00,2024-11-04 10:45:00-07:00,62,40,1,1,...,0,0.0,5.098039,5.098039,5.098039,2,2.0,2,0.0,Output/P-MARB-B124-AP-01-B111_2024-11-04_09-30...
5197,ENT 101,TNRB-151,P-TNRB-151-AP-01-140B,1,2024-11-06 14:00:00-07:00,2024-11-06 15:15:00-07:00,344,339,4,2,...,0,0.0,23.529412,23.529412,23.529412,64,64.0,64,0.0,Output/P-TNRB-151-AP-01-140B_2024-11-06_14-00_...
4105,ECON 380,MARB-B124,P-MARB-B124-AP-01-B111,2,2024-11-06 08:00:00-07:00,2024-11-06 09:15:00-07:00,62,32,1,2,...,0,0.0,5.098039,5.098039,5.098039,33,33.0,33,0.0,Output/P-MARB-B124-AP-01-B111_2024-11-06_08-00...
2969,MMBIO 121,MARB-221,P-MARB-221-AP-04-B111,1,2024-11-04 10:00:00-07:00,2024-11-04 10:50:00-07:00,267,123,4,2,...,0,0.0,5.490196,5.490196,5.490196,21,21.0,21,0.0,Output/P-MARB-221-AP-04-B111_2024-11-04_10-00_...
5152,FIN 201,TNRB-151,P-TNRB-151-AP-01-140B,1,2024-11-05 12:30:00-07:00,2024-11-05 13:45:00-07:00,344,336,4,1,...,0,0.0,38.823529,38.823529,38.823529,71,39.5,8,0.0,Output/P-TNRB-151-AP-01-140B_2024-11-05_12-30_...
5558,ACC 525,TNRB-174,P-TNRB-174-AP-01-140B,1,2024-11-06 14:00:00-07:00,2024-11-06 15:15:00-07:00,51,40,1,1,...,0,0.0,19.607843,19.607843,19.607843,40,21.0,2,0.0,Output/P-TNRB-174-AP-01-140B_2024-11-06_14-00_...
5529,ACC 525,TNRB-174,P-TNRB-174-AP-01-140B,1,2024-11-04 09:30:00-07:00,2024-11-04 10:45:00-07:00,51,39,1,1,...,0,0.0,17.647059,17.647059,17.647059,24,12.0,0,0.0,Output/P-TNRB-174-AP-01-140B_2024-11-04_09-30_...
4106,ECON 450,MARB-B124,P-MARB-B124-AP-01-B111,1,2024-11-06 11:00:00-07:00,2024-11-06 12:15:00-07:00,62,15,1,1,...,0,0.0,4.313725,4.313725,4.313725,17,8.5,0,0.0,Output/P-MARB-B124-AP-01-B111_2024-11-06_11-00...
5268,IS 201,TNRB-151,P-TNRB-151-AP-04-140B,2,2024-11-11 09:30:00-07:00,2024-11-11 10:45:00-07:00,344,277,4,2,...,0,0.0,31.764706,20.000000,20.000000,48,48.0,47,0.0,Output/P-TNRB-151-AP-04-140B_2024-11-11_09-30_...


In [35]:
print("rows in the class_summary_df represent a single 5GHz radio per class session")
print(f"total number of rows: {len(class_summary_df)}")
print(f"median number of beacons: {class_summary_df['total_beacons'].median()}")

print(f"median row of high channel utilization duration: {class_summary_df['longest_duration_high_cu'].median()}")
print(f"channel utilization is in percentage: {(class_summary_df['highest_cu'].max())}")
print(f"number of rows with total beacons less than 200: {((class_summary_df['total_beacons'] < 200).sum())}")
print(f"number of rows with channel some high channel use: {((class_summary_df['highest_cu'] > 70).sum())}")
# devices per enrolled is going to be complicated

rows in the class_summary_df represent a single 5GHz radio per class session
total number of rows: 6167
median number of beacons: 5953.0
median row of high channel utilization duration: 1.024186
channel utilization is in percentage: 100.0
number of rows with total beacons less than 200: 163
number of rows with channel some high channel use: 3509


In [ ]:
#debug
# # Check all variables in the global scope and filter DataFrames
dataframes = {name: obj for name, obj in globals().items() if isinstance(obj, pd.DataFrame)}

# Print the names of the DataFrames
print("DataFrames in memory:", list(dataframes.keys()))

# Print the head of each DataFrame
for name, df in dataframes.items():
    print(f"DataFrame: {name}")
    print(df.head(), "\n")  # Print the first 5 rows
